In [ ]:
import os
import pandas as pd
import numpy as np
import pylab as plt
import seaborn as sns
from adjustText import adjust_text

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black
jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

### Subgroup Analysis on Countries

In [ ]:
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]
colors = dict(zip(sections, sns.color_palette("colorblind", len(sections))))

In [ ]:
VERSION = "research-article_aimrd_f"
BASELINE = "baseline_2026-01-23"
RESULTS_PATH = os.path.join("../data/results/", BASELINE, VERSION)
secs = next(os.walk(RESULTS_PATH))[1]

In [ ]:
freqs_dfs = {}

for sec in secs:
    freqs_dfs[sec] = utils.get_country_frequency(RESULTS_PATH, sec)
    p = freqs_dfs[sec]["projection"]
    q = freqs_dfs[sec]["frequency"]
    totals = freqs_dfs[sec]["paper_counts"]
    reg_se = freqs_dfs[sec]["regression se"]
    freqs_dfs[sec]["regression se (ds)"] = list(
        map(utils.se, p, q, totals, ["regression"] * len(p), reg_se)
    )

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["cutoff"] == "english"]

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["cutoff"] == "non-english"]

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["cutoff"] == "Canada"]

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["cutoff"] == "Taiwan"]

In [ ]:
country_alt_names = {
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "UK",
    "Korea, Republic of": "South Korea",
    "Iran, Islamic Republic of": "Iran",
}

In [ ]:
df = pd.concat(freqs_dfs, names=["section"]).reset_index(level=0).reset_index(drop=True)
df["country"] = [
    country_alt_names[c] if c in country_alt_names.keys() else c for c in df["cutoff"]
]
df = df.drop("cutoff", axis=1)
df["section"] = pd.Categorical(
    df["section"],
    categories=[
        "abstract",
        "introduction",
        "methods",
        "results",
        "discussion",
        "full",
    ],
    ordered=True,
)
df

In [ ]:
exclude = ["Türkiye", "Poland"]
df_filtered = df[
    (df["time"].between(2022, 2025))
    & (df["section"].isin(["abstract", "full"]))
    & (~df["country"].isin(exclude))
]
df_filtered

In [ ]:
countries = df_filtered["country"].unique()
n = len(countries)

ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols, nrows),
    sharex=True,
    sharey=True,
    layout="constrained",
)
axes = axes.flatten()

for i, country in enumerate(countries):
    ax = axes[i]
    sub = df_filtered[df_filtered["country"] == country]

    for section in ["abstract", "full"]:
        sec_data = sub[sub["section"] == section].sort_values("time")

        ax.errorbar(
            sec_data["time"],
            sec_data["usage estimate"],
            yerr=sec_data["regression se (ds)"],
            color=colors[section],
            marker="o",
            linestyle="-",
            capsize=2,
            markersize=2.5,
            clip_on=True,
            label=section,
        )

    # Title
    ax.set_title(country)

    # Max usage annotation
    max_usage = sub["usage estimate"].max()
    ax.text(
        0.75,  # 0.05,
        0.05,  # 0.9,
        f"{max_usage:.2f}",
        transform=ax.transAxes,
        bbox=dict(
            facecolor="#666666", edgecolor="none", alpha=0.5, boxstyle="round,pad=.2"
        ),
    )

    ax.set_ylim([0, 1])

    if i % ncols == 0:
        ax.set_ylabel("LLM-use")

axes[0].legend()

# plt.tight_layout()
plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "countries_individual.png",
    ),
    dpi=300,
)
plt.show()

In [ ]:
df_full = df[(df["section"] == "full") & (df["time"].isin([2024, 2025]))]
df_full = df_full.pivot(
    index="country", columns="time", values="usage estimate"
).reset_index()

In [ ]:
english_countries = ["USA", "UK", "Canada", "Australia", "english"]
df_full["group"] = [
    "english" if c in english_countries else "non-english" for c in df_full["country"]
]
df_full

In [ ]:
fig, ax = plt.subplots(
    figsize=(3, 3),
    layout="constrained",
)

for group, d in df_full.groupby("group"):
    ax.scatter(d[2024], d[2025], label=group)

texts = []
for _, row in df_full.iterrows():
    texts.append(ax.text(row[2024], row[2025], row["country"], fontsize=5))

adjust_text(texts, ax=ax)

ax.set_xlabel("LLM-use (2024)")
ax.set_ylabel("LLM-use (2025)")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.legend()

plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "countries_2024v2025.png",
    ),
    dpi=300,
)
plt.show()

In [ ]:
df_filtered = df[(df["section"] == "full") & (~df["country"].isin(exclude))]

countries = df_filtered["country"].unique()
n = len(countries)

ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols, nrows),
    sharex=True,
    sharey=True,
    layout="constrained",
)
axes = axes.flatten()

for i, country in enumerate(countries):
    ax = axes[i]
    sub = df_filtered[df_filtered["country"] == country].sort_values("time")

    ax.plot(
        sub["time"],
        sub["frequency"],
        color=colors["full"],
        label="($q$)",
    )

    ax.plot(
        sub["time"],
        sub["projection"],
        # yerr=sub["regression se"],
        color=colors["full"],
        linestyle="--",
        label="($\\hat{p}_{human}$)",
    )

    ax.fill_between(
        sub["time"],
        sub["projection"] - sub["regression se"],
        sub["projection"] + sub["regression se"],
        alpha=0.2,
        color=colors["full"],
    )

    # Title
    ax.set_title(country)

    ax.set_ylim([0.6, 1])
    ax.set_xticks(range(2019, 2026, 2))

    if i % ncols == 0:
        ax.set_ylabel("Frequency")
    if i >= ncols * (nrows - 1):
        ax.set_xticklabels(range(2019, 2026, 2), rotation=60)


axes[0].legend()

# plt.tight_layout()
plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "countries_individual_freqs.png",
    ),
    dpi=300,
)
plt.show()